In [1]:
import pandas as pd

base_dir = "C:/Users/Coelho/Desktop/joao/TB/medicamentos-fdcc/tasks"

itens  = pd.read_csv(f"{base_dir}/coleta-contratacoes/outputs/2025/20261104_123936_contratacoes_2025/df/itens.csv")
adjudicacoes = pd.read_csv(f"{base_dir}/coleta-contratacoes/outputs/2025/20261104_123936_contratacoes_2025/df/adjudicacoes.csv")
contratacoes = pd.read_csv(f"{base_dir}/coleta-contratacoes/outputs/2025/20261104_123936_contratacoes_2025/df/contratacoes.csv")

contratos_anual = pd.read_csv(f"{base_dir}/coleta-contratos/outputs/anual/2025/20261103_170224_contratos_anual_2025.csv")

C:\Users\Coelho\AppData\Local\Temp\ipykernel_4156\1119313309.py:9: DtypeWarning: Columns (22: fundamento_legal) have mixed types. Specify dtype option on import or set low_memory=False.
  contratos_anual = pd.read_csv(f"{base_dir}/coleta-contratos/outputs/anual/2025/20261103_170224_contratos_anual_2025.csv")


In [2]:
# União de itens + adjudicações + contratações
# Chaves: itens <-> adjudicacoes por (ocid, item_id, lote_id); <-> contratacoes por ocid
chaves_item = ["ocid", "item_id", "lote_id"]

# Colunas sobrepostas entre itens e adjudicações (fora as chaves):
# - unidade: idêntica nos dois -> mantém só a de itens
# - quantidade: em adjudicações é a quantidade adjudicada (pode diferir da estimada) -> renomeia
adjudicacoes_join = (
    adjudicacoes
    .drop(columns=["unidade"])
    .rename(columns={"quantidade": "quantidade_adjudicada"})
)

# Left join: mantém itens sem adjudicação; um item pode ter mais de uma adjudicação
contratacoes_completo = (
    itens
    .merge(adjudicacoes_join, on=chaves_item, how="left", validate="one_to_many")
    .merge(contratacoes, on="ocid", how="left", validate="many_to_one")
)

# Garante que nenhuma coluna ficou repetida (sem sufixos _x/_y)
assert contratacoes_completo.columns.is_unique
assert not contratacoes_completo.columns.str.endswith(("_x", "_y")).any()

print(contratacoes_completo.shape)
contratacoes_completo.head(50)

(2570, 57)


,ocid,item_id,lote_id,descricao,codigo_catalogo,esquema_classificacao,uri_catalogo,categoria_id,categoria_nome,ncm_nbs,...,metodo_contratacao_detalhes,amparo_legal,valor_estimado,moeda,data_inicio_propostas,data_fim_propostas,forma_submissao,qtd_itens,qtd_adjudicacoes,valor_adjudicado_total
0,ocds-ye9ov3-11080085000181_2025_3,1,lot-1,Álcool Etílico,445578,CATMAT,https://cnbs.estaleiro.serpro.gov.br/cnbs-api/...,NaN,NaN,NaN,...,tipoInstrumentoConvocatorioCodigo: 2; tipoInst...,amparoLegalCodigo: 19; amparoLegalNome: Lei 14...,6.419000e+02,BRL,2025-03-10T17:56:20Z,2025-03-19T10:59:59Z,NaN,1,1,532.70
1,ocds-ye9ov3-15126437000305_2025_2200,1,lot-1,Acetazolamida,278283,CATMAT,https://cnbs.estaleiro.serpro.gov.br/cnbs-api/...,NaN,NaN,30049079.0,...,tipoInstrumentoConvocatorioCodigo: 1; tipoInst...,amparoLegalCodigo: 1; amparoLegalNome: Lei 14....,0.000000e+00,BRL,2025-08-19T11:00:00Z,2025-09-25T13:00:00Z,electronicSubmission,8,5,179250.00
2,ocds-ye9ov3-15126437000305_2025_2200,2,lot-2,Ácido fólico,267503,CATMAT,https://cnbs.estaleiro.serpro.gov.br/cnbs-api/...,NaN,NaN,30045090.0,...,tipoInstrumentoConvocatorioCodigo: 1; tipoInst...,amparoLegalCodigo: 1; amparoLegalNome: Lei 14....,0.000000e+00,BRL,2025-08-19T11:00:00Z,2025-09-25T13:00:00Z,electronicSubmission,8,5,179250.00
3,ocds-ye9ov3-15126437000305_2025_2200,3,lot-3,Ácido tranexâmico,384830,CATMAT,https://cnbs.estaleiro.serpro.gov.br/cnbs-api/...,NaN,NaN,30049039.0,...,tipoInstrumentoConvocatorioCodigo: 1; tipoInst...,amparoLegalCodigo: 1; amparoLegalNome: Lei 14....,0.000000e+00,BRL,2025-08-19T11:00:00Z,2025-09-25T13:00:00Z,electronicSubmission,8,5,179250.00
4,ocds-ye9ov3-15126437000305_2025_2200,4,lot-4,Ácido tranexâmico,384830,CATMAT,https://cnbs.estaleiro.serpro.gov.br/cnbs-api/...,NaN,NaN,30049039.0,...,tipoInstrumentoConvocatorioCodigo: 1; tipoInst...,amparoLegalCodigo: 1; amparoLegalNome: Lei 14....,0.000000e+00,BRL,2025-08-19T11:00:00Z,2025-09-25T13:00:00Z,electronicSubmission,8,5,179250.00
5,ocds-ye9ov3-15126437000305_2025_2200,6,lot-6,Alfaepoetina,449016,CATMAT,https://cnbs.estaleiro.serpro.gov.br/cnbs-api/...,NaN,NaN,30043929.0,...,tipoInstrumentoConvocatorioCodigo: 1; tipoInst...,amparoLegalCodigo: 1; amparoLegalNome: Lei 14....,0.000000e+00,BRL,2025-08-19T11:00:00Z,2025-09-25T13:00:00Z,electronicSubmission,8,5,179250.00
6,ocds-ye9ov3-15126437000305_2025_2200,7,lot-7,Alprostadil,268380,CATMAT,https://cnbs.estaleiro.serpro.gov.br/cnbs-api/...,NaN,NaN,30043999.0,...,tipoInstrumentoConvocatorioCodigo: 1; tipoInst...,amparoLegalCodigo: 1; amparoLegalNome: Lei 14....,0.000000e+00,BRL,2025-08-19T11:00:00Z,2025-09-25T13:00:00Z,electronicSubmission,8,5,179250.00
7,ocds-ye9ov3-15126437000305_2025_2200,8,lot-8,Alprostadil,268380,CATMAT,https://cnbs.estaleiro.serpro.gov.br/cnbs-api/...,NaN,NaN,30043999.0,...,tipoInstrumentoConvocatorioCodigo: 1; tipoInst...,amparoLegalCodigo: 1; amparoLegalNome: Lei 14....,0.000000e+00,BRL,2025-08-19T11:00:00Z,2025-09-25T13:00:00Z,electronicSubmission,8,5,179250.00
8,ocds-ye9ov3-15126437000305_2025_2200,10,lot-10,Amiodarona,271709,CATMAT,https://cnbs.estaleiro.serpro.gov.br/cnbs-api/...,NaN,NaN,30049054.0,...,tipoInstrumentoConvocatorioCodigo: 1; tipoInst...,amparoLegalCodigo: 1; amparoLegalNome: Lei 14....,0.000000e+00,BRL,2025-08-19T11:00:00Z,2025-09-25T13:00:00Z,electronicSubmission,8,5,179250.00
9,ocds-ye9ov3-15126437000305_2025_253,1,lot-1,"Carboplatina Carboplatina Dosagem: 150MG, Uso:...",270411,CATMAT,https://cnbs.estaleiro.serpro.gov.br/cnbs-api/...,NaN,NaN,NaN,...,tipoInstrumentoConvocatorioCodigo: 3; tipoInst...,amparoLegalCodigo: 21; amparoLegalNome: Lei 14...,3.857158e+06,BRL,NaN,NaN,NaN,9,9,3454505.16


In [3]:
contratos_anual

,id,receita_despesa,numero,orgao_codigo,orgao_nome,unidade_codigo,esfera,poder,sisg,gestao,...,data_assinatura,data_publicacao,vigencia_inicio,vigencia_fim,valor_inicial,valor_global,num_parcelas,valor_parcela,valor_acumulado,situacao
0,434816,Despesa,2022NE003590,26244,UNIVERSIDADE FEDERAL DO RIO GRANDE DO SUL,153114,Federal,Executivo,Sim,15235,...,2022-11-18,2022-11-18,2022-11-18,2022-12-31,11604.00,11604.00,1,11604.00,0.0,Ativo
1,434427,Despesa,2022NE003177,26244,UNIVERSIDADE FEDERAL DO RIO GRANDE DO SUL,153114,Federal,Executivo,Sim,15235,...,2022-10-21,2022-10-21,2022-10-21,2022-12-31,676.80,676.80,1,676.80,0.0,Ativo
2,434734,Despesa,2022NE003497,26244,UNIVERSIDADE FEDERAL DO RIO GRANDE DO SUL,153114,Federal,Executivo,Sim,15235,...,2022-11-10,2022-11-10,2022-11-10,2022-12-31,5730.40,5730.40,1,5730.40,0.0,Ativo
3,434736,Despesa,2022NE003506,26244,UNIVERSIDADE FEDERAL DO RIO GRANDE DO SUL,153114,Federal,Executivo,Sim,15235,...,2022-11-10,2022-11-10,2022-11-10,2022-12-31,108000.00,108000.00,1,108000.00,0.0,Ativo
4,434759,Despesa,2022NE003540,26244,UNIVERSIDADE FEDERAL DO RIO GRANDE DO SUL,153114,Federal,Executivo,Sim,15235,...,2022-11-11,2022-11-11,2022-11-11,2022-12-31,1598.00,1598.00,1,1598.00,0.0,Ativo
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
279825,610689,Despesa,2025NE000078,26402,"INST.FED.DE EDUC.,CIENC.E TEC.DE ALAGOAS",152805,Federal,Executivo,Sim,26402,...,2025-07-07,2025-07-07,2025-07-07,2025-12-31,233.60,233.60,1,233.60,233.6,Ativo
279826,610659,Despesa,2025NE000488,26231,UNIVERSIDADE FEDERAL DE ALAGOAS,153037,Federal,Executivo,Sim,15222,...,2025-07-17,NaN,2025-07-17,2025-12-31,850.00,850.00,1,850.00,850.0,Ativo
279827,610666,Despesa,2025NE000697,95556,ESP-SECRETARIA DA SAUDE,90183,Estadual,Executivo,Não,95556,...,2025-07-30,NaN,2025-08-05,2025-08-20,1134.90,1134.90,1,1134.90,1134.9,Ativo
279828,522123,Despesa,2025NE000587,36000,MINISTERIO DA SAUDE,250103,Federal,Executivo,Sim,1,...,2025-04-14,NaN,2025-04-14,2025-12-31,875.00,875.00,1,875.00,0.0,Ativo


In [4]:
# Cruzamento de contratacoes_completo com contratos_anual
#
# contratos_anual (Contratos.gov.br) não tem ocid. Montamos o identificador da compra
# a partir de unidade_compra, codigo_modalidade e licitacao_numero ("00305/2022").
# O cruzamento é feito por (ocid, fornecedor): cada item adjudicado recebe o(s) contrato(s)
# firmados com o fornecedor vencedor naquela compra.
prefixo_ocid = "ocds-914jxj-"

def so_digitos(s):
    return s.astype("string").str.replace(r"\D", "", regex=True).replace("", pd.NA)

numero_ano = contratos_anual["licitacao_numero"].astype("string").str.extract(r"^(\d+)/(\d{4})$")
uasg = pd.to_numeric(contratos_anual["unidade_compra"], errors="coerce").astype("Int64").astype("string").str.zfill(6)
modalidade = contratos_anual["codigo_modalidade"].astype("string").str.zfill(2)

contratos_join = (
    contratos_anual
    .assign(
        ocid=prefixo_ocid + uasg + modalidade + numero_ano[0].str.zfill(5) + numero_ano[1],
        fornecedor_chave=so_digitos(contratos_anual["fonecedor_cnpj_cpf_idgener"]),
    )
    # fornecedor já está identificado pela chave; nome vem de adjudicações
    .drop(columns=["fonecedor_cnpj_cpf_idgener", "fornecedor_nome"])
    .dropna(subset=["ocid", "fornecedor_chave"])
)
# Prefixa as colunas de contrato para não colidir com as de contratações (objeto, situacao, esfera...)
contratos_join = contratos_join.rename(
    columns={c: f"contrato_{c}" for c in contratos_join.columns if c not in ("ocid", "fornecedor_chave")}
)

# Left join: mantém todos os itens; itens sem contrato ficam com colunas contrato_* vazias.
# Um (ocid, fornecedor) pode ter mais de um contrato -> a linha do item se repete por contrato.
contratacoes_contratos = (
    contratacoes_completo
    .assign(fornecedor_chave=so_digitos(contratacoes_completo["fornecedor_id"]))
    .merge(contratos_join, on=["ocid", "fornecedor_chave"], how="left", indicator="_cruzamento")
    .drop(columns=["fornecedor_chave"])
)

assert contratacoes_contratos.columns.is_unique
assert not contratacoes_contratos.columns.str.endswith(("_x", "_y")).any()

# Diagnóstico do cruzamento
ocids_base = set(contratacoes_completo["ocid"])
contratos_da_base = contratos_join[contratos_join["ocid"].isin(ocids_base)]
print(f"Linhas: {len(contratacoes_completo):,} -> {len(contratacoes_contratos):,}")
print(f"Linhas com contrato: {(contratacoes_contratos['_cruzamento'] == 'both').sum():,}")
print(f"Contratações com ao menos um contrato: {contratacoes_contratos.loc[contratacoes_contratos['_cruzamento'] == 'both', 'ocid'].nunique():,} de {len(ocids_base):,}")
print(f"Contratos cuja compra está na base: {len(contratos_da_base):,} | casados também por fornecedor: {contratos_da_base['contrato_id'].isin(contratacoes_contratos['contrato_id']).sum():,}")

contratacoes_contratos = contratacoes_contratos.drop(columns=["_cruzamento"])
contratacoes_contratos

Linhas: 2,570 -> 2,570
Linhas com contrato: 0
Contratações com ao menos um contrato: 0 de 683
Contratos cuja compra está na base: 0 | casados também por fornecedor: 0


,ocid,item_id,lote_id,descricao,codigo_catalogo,esquema_classificacao,uri_catalogo,categoria_id,categoria_nome,ncm_nbs,...,contrato_data_assinatura,contrato_data_publicacao,contrato_vigencia_inicio,contrato_vigencia_fim,contrato_valor_inicial,contrato_valor_global,contrato_num_parcelas,contrato_valor_parcela,contrato_valor_acumulado,contrato_situacao
0,ocds-ye9ov3-11080085000181_2025_3,1,lot-1,Álcool Etílico,445578,CATMAT,https://cnbs.estaleiro.serpro.gov.br/cnbs-api/...,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ocds-ye9ov3-15126437000305_2025_2200,1,lot-1,Acetazolamida,278283,CATMAT,https://cnbs.estaleiro.serpro.gov.br/cnbs-api/...,NaN,NaN,30049079.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ocds-ye9ov3-15126437000305_2025_2200,2,lot-2,Ácido fólico,267503,CATMAT,https://cnbs.estaleiro.serpro.gov.br/cnbs-api/...,NaN,NaN,30045090.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,ocds-ye9ov3-15126437000305_2025_2200,3,lot-3,Ácido tranexâmico,384830,CATMAT,https://cnbs.estaleiro.serpro.gov.br/cnbs-api/...,NaN,NaN,30049039.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,ocds-ye9ov3-15126437000305_2025_2200,4,lot-4,Ácido tranexâmico,384830,CATMAT,https://cnbs.estaleiro.serpro.gov.br/cnbs-api/...,NaN,NaN,30049039.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2565,ocds-ye9ov3-8829974000194_2025_118,6,lot-6,Bromoprida Bromoprida Dosagem: 10 M,269954,CATMAT,https://cnbs.estaleiro.serpro.gov.br/cnbs-api/...,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2566,ocds-ye9ov3-8829974000194_2025_118,7,lot-7,Escopolamina Butilbrometo Escopolamina Butilbr...,267281,CATMAT,https://cnbs.estaleiro.serpro.gov.br/cnbs-api/...,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2567,ocds-ye9ov3-8829974000194_2025_118,8,lot-8,Cefalexina Cefalexina Dosagem: 500 M,267625,CATMAT,https://cnbs.estaleiro.serpro.gov.br/cnbs-api/...,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2568,ocds-ye9ov3-8829974000194_2025_118,9,lot-9,"Ceftazidima Ceftazidima Concentração: 1 G, For...",442694,CATMAT,https://cnbs.estaleiro.serpro.gov.br/cnbs-api/...,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
# Filtro 1: somente linhas que casaram com algum contrato
contratacoes_com_contrato = contratacoes_contratos[contratacoes_contratos["contrato_id"].notna()].copy()

print(f"Linhas: {len(contratacoes_com_contrato):,}")
print(f"Contratações (ocid): {contratacoes_com_contrato['ocid'].nunique():,}")
print(f"Contratos distintos: {contratacoes_com_contrato['contrato_id'].nunique():,}")
contratacoes_com_contrato

Linhas: 0
Contratações (ocid): 0
Contratos distintos: 0


,ocid,item_id,lote_id,descricao,codigo_catalogo,esquema_classificacao,uri_catalogo,categoria_id,categoria_nome,ncm_nbs,...,contrato_data_assinatura,contrato_data_publicacao,contrato_vigencia_inicio,contrato_vigencia_fim,contrato_valor_inicial,contrato_valor_global,contrato_num_parcelas,contrato_valor_parcela,contrato_valor_acumulado,contrato_situacao
